In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh")

print("Path to dataset files:", path)

In [ ]:
import os

# Explore the folder structure
dataset_folder = path  # The path to your downloaded dataset
print(f"Dataset folder: {dataset_folder}")
os.listdir(dataset_folder)

In [ ]:
import numpy as np
from PIL import Image
import cv2  # For image handling

# Define image size
common_size = (256, 256)

images = []
labels = []

# Walk through the dataset folder
for root, dirs, files in os.walk(dataset_folder):
    for file_name in files:
        if file_name.lower().endswith('.jpg') or file_name.lower().endswith('.png'):  # Checking if it's an image
            image_path = os.path.join(root, file_name)
            try:
                # Load and resize image
                image = Image.open(image_path)
                image = image.resize(common_size)
                image_data = np.array(image)

                # Add image data
                images.append(image_data)

                # Add the label (folder name as label)
                label = os.path.basename(root)
                labels.append(label)

            except Exception as e:
                print(f"Could not load {file_name}: {e}")

# Convert lists to numpy arrays
images = np.array(images)
labels = np.array(labels)

print(f"Total images loaded: {len(images)}")

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Encode labels
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded)

print(f"Encoded labels: {labels_encoded}")
print(f"Categorical labels: {labels_categorical}")

In [ ]:
from sklearn.model_selection import train_test_split

# Normalize pixel values to be between 0 and 1
images = images / 255.0

# Split data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(images, labels_categorical, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}, Testing set: {X_test.shape}")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(np.unique(labels_encoded)), activation='softmax'))  # Output layer with number of classes

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
print(model.summary())

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=10, batch_size=32)

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot accuracy
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='upper left')
plt.show()

# Plot loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper left')
plt.show()

In [ ]:
predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)

print(f"Predicted classes: {predicted_classes}")
print(f"True classes: {true_classes}")

In [ ]:
model.save("pumpkin_disease_model.h5")

The below block of code can be used as a full fledged model for prediction.

In [ ]:
from tensorflow.keras.models import load_model

# Load the saved model
model = load_model("pumpkin_disease_model.h5")

# Predict a new image (make sure it's preprocessed similarly)
new_image = Image.open('path_to_new_image.jpg').resize(common_size)
new_image = np.array(new_image) / 255.0
new_image = new_image.reshape(1, 256, 256, 3)  # Add batch dimension

prediction = model.predict(new_image)
predicted_class = np.argmax(prediction, axis=1)

print(f"Predicted class: {label_encoder.inverse_transform(predicted_class)}")